
Projet : Détection d'Intrusion (Reconnaissance et Scanning)

 **Objectif :** Développer un modèle de Machine Learning / Deep Learning pour distinguer le comportement réseau normal (Benign) des activités de reconnaissance et de scan, en utilisant des données issues de fichiers JSON et un dataset normal.

### 0.1 Importation des bibliothèques
 Importez ici les outils nécessaires : manipulation de données (pandas, numpy), visualisation (matplotlib, seaborn), ML/DL (sklearn, tensorflow/keras).


In [2]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix, roc_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.utils import to_categorical

In [6]:
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense
    print("✅ Import réussi !")
except Exception as e:
    print("❌ Erreur :", e)

✅ Import réussi !


 ## 1. Phase 1 : Exploration des Données (EDA)
 ### 1.1 Chargement des Données Normales (Benign) et d'Attaque (Reconnaissance)



In [ ]:
# %% [markdown]
# ## 🛠️ Correction de la Phase 1.1 : Gestion des Fichiers JSON Mal Formés
#
# L'erreur `Extra data` indique que les fichiers contiennent plusieurs objets JSON non séparés. Nous devons lire et décoder ligne par ligne pour gérer cette structure.

# %%
import pandas as pd
import json
import os

# --- Définition des chemins (réutilisés) ---
PATH_ATTACK_DIR = r'C:\Users\User\Downloads\projet python\recon'
PATH_BENIGN_DIR = r'C:\Users\User\Downloads\projet python\benign' 


# Fonction utilitaire CORRIGÉE pour charger et étiqueter les fichiers JSON
def load_and_merge_json_data_fixed(directory_path, label):
    """
    Charge tous les fichiers .json en lisant ligne par ligne 
    pour gérer les objets JSON concaténés (Extra data error).
    """
    data_list = []
    
    try:
        json_files = [f for f in os.listdir(directory_path) if f.endswith('.json')]
    except FileNotFoundError:
        print(f"Erreur: Le chemin du répertoire pour '{label}' est introuvable : {directory_path}")
        return pd.DataFrame()
        
    total_files = len(json_files)
    print(f"Chargement de {total_files} fichiers JSON pour la catégorie '{label}'...")

    for i, file_name in enumerate(json_files):
        file_path = os.path.join(directory_path, file_name)
        
        # Affichage de la progression
        if (i + 1) % 100 == 0 or i == total_files - 1:
            print(f"Progression ({label}) : {i + 1}/{total_files} fichiers traités.")
            
        try:
            # Ouvre le fichier et lit tout son contenu
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            # Tente d'abord le chargement simple (cas où le fichier est un tableau JSON standard [])
            try:
                data = json.loads(content)
            except json.JSONDecodeError:
                # Si le chargement simple échoue, tente de charger les objets JSON multiples concaténés
                # Ceci est l'étape critique pour l'erreur "Extra data"
                decoder = json.JSONDecoder()
                pos = 0
                data = []
                while pos < len(content):
                    try:
                        obj, pos = decoder.raw_decode(content, pos)
                        if isinstance(obj, list): # Si un tableau est trouvé
                            data.extend(obj)
                        else:
                            data.append(obj)
                    except json.JSONDecodeError:
                        # Si l'objet n'est pas trouvé (caractère non-JSON), avance d'un caractère
                        pos += 1 
                        
            # Normalisation : s'assurer que 'data' est une liste d'enregistrements
            if isinstance(data, dict):
                data = [data]
            elif not isinstance(data, list):
                continue
                
            # Ajout de l'étiquette
            for record in data:
                record['Label'] = label
                data_list.append(record)
                    
        except Exception as e:
            print(f"Avertissement: Erreur inattendue lors du traitement de {file_name} ({label}): {e}. Fichier ignoré.")
            
    return pd.DataFrame(data_list)

# %% [markdown]
# ### 1.3 Exécution du Chargement Corrigé

# %%
# 1. Chargement des données d'attaque (CORRIGÉ)
df_attack = load_and_merge_json_data_fixed(PATH_ATTACK_DIR, 'Attack')
print(f"\nDataFrame d'attaque créé. Taille : {df_attack.shape}")

# 2. Chargement des données normales (Benign) - Assurez-vous que ce chemin est correct.
df_benign = load_and_merge_json_data_fixed(PATH_BENIGN_DIR, 'Benign') 
print(f"\nDataFrame Bénin créé. Taille : {df_benign.shape}")

# 3. Fusion des deux datasets
if not df_attack.empty or not df_benign.empty:
    # Utiliser concat même si un est vide, mais imprimer un avertissement
    df_raw = pd.concat([df_benign, df_attack], ignore_index=True)
    print(f"\nDataset brut total fusionné : {df_raw.shape}")
    print(f"Distribution finale des classes : \n{df_raw['Label'].value_counts()}")
else:
    print("\nÉchec total du chargement des données. Vérifiez les chemins et le format des fichiers.")
    df_raw = pd.DataFrame()

Chargement de 233 fichiers JSON pour la catégorie 'Attack'...
Progression (Attack) : 100/233 fichiers traités.
Progression (Attack) : 200/233 fichiers traités.
Progression (Attack) : 233/233 fichiers traités.

DataFrame d'attaque créé. Taille : (290089, 4)
Chargement de 1 fichiers JSON pour la catégorie 'Benign'...
Progression (Benign) : 1/1 fichiers traités.



 ## 2. Phase 2 : Préparation des Données

 Le chargement ayant réussi, nous passons à la préparation, en commençant par la standardisation des noms de colonnes et la gestion des types de données.




In [ ]:

# Si l'exécution précédente n'a pas affiché la taille de df_benign, 
# vous pouvez l'afficher ici avant la fusion.
print(f"Taille du DataFrame Bénin : {df_benign.shape}")
# --- Fusion des deux DataFrames (si ce n'est pas déjà fait) ---
if 'df_raw' not in locals() or df_raw.empty:
    if not df_attack.empty or not df_benign.empty:
        df_raw = pd.concat([df_benign, df_attack], ignore_index=True)
        print(f"\nDataset brut total fusionné : {df_raw.shape}")
        print(f"Distribution finale des classes : \n{df_raw['Label'].value_counts()}")
    else:
        print("Erreur: DataFrames d'attaque et bénin sont vides.")

# Inspection des colonnes
print("\n Aperçu des colonnes après fusion :")
print(df_raw.columns)
print(f"Nombre total de colonnes uniques : {len(df_raw.columns)}")